# GraphRAG: AI Copyright & Governance Knowledge Graph. 
This notebook builds a full **GraphRAG pipeline** on a pre-scraped dataset of news articles,  
about AI copyright, governance, and intellectual property.
### Pipeline overview
| Step | What happens |
|---|---|
| 1 | Load and prepare your scraped article dataset.|
| 2 | Define a domain ontology for AI copyright/governance.|
| 3 | Extract entities + relationships WITH descriptions using a custom extractor.|
| 4 | Build a knowledge graph with community detection baked in.|
| 5 | Generate LLM community summaries.|
| 6 | Visualize the graph interactively.|
| 7 | Query using GraphRAG |

In [6]:
import asyncio
import nest_asyncio
import pandas as pd
import networkx as nx 
from dotenv import load_dotenv 
from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.graph_stores.types import (
    EntityNode, KG_NODES_KEY , KG_RELATIONS_KEY , Relation
)
from llama_index.core.graph_stores import SimplePropertyGraphStore 
from llama_index.core.llms.llm import LLM 
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import TransformComponent, BaseNode 
from llama_index.core.async_utils import run_jobs 
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.llms.openai import OpenAI
from graspologic.partition import hierarchical_leiden
# Patch the event loop so LlamaIndex async calls work inside Jupyter
nest_asyncio.apply() 
load_dotenv() 
print("Import Ready")

Import Ready


In [14]:
LLM = OpenAI(model = "gpt-4o-mini")
# -- Pipeline Parameters --------------------------------------------
MAX_ARTICLES = 50
MAX_PATH_PER_CHUNK = 20 
NUM_WORKERS = 4
MAX_CLUSTER_SIZE = 10 
GRAPH_OUTPUT_FILE = "ai_copyright_graph.html" 
Settings.llm = LLM 
print(f"Use {LLM.model} in this directory.")

Use gpt-4o-mini in this directory.


## 3. Define the Ontology 
The ontology is the schema of our Knowledge Graph. It controls exactly what kinds of entities and relationships the LLM is allowed to extract.  
### Why it matters
Without a defined schema, the LLM will:
* Invest differnt entity types per document ("AI_COMPANY" vs "TECH_FIRM" vs "ORGANIZATION").
* Use inconsistent relationship labels for the same concepts.
* Create duplicate node for the same entity under slightly different names.

The ontology constrains extraction to a fixed vocabulary, so the graph stays consistent across all documents. 
### For this particular use case - AI copyright & governance.
I defined below 7 entity types, 8 relationship type.  
But this really depends on your use case and what you want to research.

In [2]:
# -- Entity types ------------------
# We'll embed these directly into the extraction prompt below.

ENTITY_TYPES = [
    "ORGANISATION",     # Companies, labs, industry groups (OpenAI, Google, RIAA) 
    "PERSON",           # Executives, policymakers, judges (Sam Altman, Thierry Breton) 
    "LEGISLATION",      # Laws, acts, regulations (EU AI Act, DMCA, Copyright Act) 
    "LEGAL_CASE",       # Lawsuits, court rulings (NYT v. OpenAI, Getty v. Stability AI) 
    "CONCEPT",          # Abstract ideas: fair use, training data, IP rights
    "GOVERNMENT",       # Nations, regulatory bodies, courts (EU, US Copyright Office) 
    "AI_SYSTEM"         # Specific models or products (GPT-4, Stable Diffusion , Gemini) 
]

# --  Relationship types ---------------
RELATION_TYPES = [
    "FILED_AGAINST",    # Plaintiff Organisation/Person -> Legal_Case 
    "DEFENDANT_IN",     # Organization/Person -> Legal_Case 
    "REGULATES",        # GOVERNMENT/LEGISLATION  -> Organization / AI_SYSTEM 
    "ADVOCATES_FOR",    # ORGANISATION/PERSON  -> CONCEPT or policy position
    "TARINED_ON",       # AI_SYSTEM -> Concept or dataset type
    "PART_OF",          # PERSON -> ORGANIZATION
    "REFRENCES",        # LEGAL_CASE /LEGISLATION -> CONCEPT 
    "OPPOSES"           # ORGANIZATION/GOVERNMENT -> LEGISLATION/CONCEPT
]
print("✅ Ontology:") 
print(f" Entity types: {ENTITY_TYPES}")
print(f" Relationship types: {RELATION_TYPES}")

✅ Ontology:
 Entity types: ['ORGANISATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM']
 Relationship types: ['FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TARINED_ON', 'PART_OF', 'REFRENCES', 'OPPOSES']


## 4. Extraction Prompt 
This is where our ontology gets embedded into the LLM's instructions.
Most GraphRAG implementations extract bare triplets: (OpenAI, DEFENDANT_IN, NYT v. OpenAI).
This prompt extracts desciptions alongside every entity and relatioship:
* Entity desciption : "OpenAI is an AI research company that developed GPT-4, currently facing multiple copyright lawsuits"
* Relationship description : "OpenAI is named as defendant in the New York Times lawsuit over  allegedly using copyrighted articles as training data. 
These descriptions flow through the community summaries, making them far richer and enabling much better final answer.

In [5]:
# Build the entity and relationship type strings dynamically from our ontology.
entity_types_str = ", ".join(ENTITY_TYPES) 
relation_types_str = ", ".join(RELATION_TYPES) 
print(entity_types_str)

KG_TRIPLET_EXTRACT_TMPL = f"""
                        -- GOAL -- 
                        Given a news article about AI copyright, governance, or intellectual property.
                        identify all entities mentioned in the article and their relationships.
                        Extract up to {{max_knowlege_triplets}} entity-relation triplets.

                        - Allowed Entity Types -
                        {entity_types_str} 

                        - Allowed Relationship Types - 
                        {relation_types_str} 

                        - Steps - 
                        1. Identify ALL entities. For each entity extract: 
                            - name : Name of the entity, capitalized
                            - type : One of the allowed entity types above
                            - description : A  brief description of the entity and its role in AI copyright/governance 
                        2. Identify relationships between entities. For each extract:
                            - source : name of the source entity
                            - target : name of the target entity 
                            - relation : one of the allowed relationship types above
                            - description : a sentence explaining why and how these entities are related.        
                        - Real Data -
                        ###################
                        text:{{text}}
                        ###################
                            
                        """
print("✅ Extraction prompt ready") 
print(f"\nPreview (first 300 chars):\n {KG_TRIPLET_EXTRACT_TMPL[:300]} . . .")


ORGANISATION, PERSON, LEGISLATION, LEGAL_CASE, CONCEPT, GOVERNMENT, AI_SYSTEM
✅ Extraction prompt ready

Preview (first 300 chars):
 
                        -- GOAL -- 
                        Given a news article about AI copyright, governance, or intellectual property.
                        identify all entities mentioned in the article and their relationships.
                        Extract up to {max_knowlege_triplets} en . . .


### Pydantic Extraction Model.